# Detecção de Fraude em Cartão de Crédito

Este notebook implementa um sistema de detecção de fraude em transações de cartão de crédito utilizando técnicas de Machine Learning.

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, f1_score
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Carregamento e Exploração dos Dados

In [ ]:
# Carregar os dados
df = pd.read_csv('data/creditcard.csv')

# Exibir informações básicas
print("Forma do dataset:", df.shape)
print("\nPrimeiras linhas:")
print(df.head())
print("\nInformações do dataset:")
print(df.info())
print("\nEstatísticas descritivas:")
print(df.describe())

## 3. Análise Exploratória de Dados (EDA)

In [ ]:
# Verificar valores faltantes
print("Valores faltantes:")
print(df.isnull().sum())

# Distribuição de classes
print("\nDistribuição de classes:")
print(df['Class'].value_counts())
print("\nPercentual:")
print(df['Class'].value_counts(normalize=True) * 100)

In [ ]:
# Visualizar desequilíbrio de classes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico de contagem
df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Distribuição de Classes (Contagem)')
axes[0].set_xlabel('Classe (0=Legítimo, 1=Fraude)')
axes[0].set_ylabel('Quantidade')
axes[0].set_xticklabels(['Legítimo', 'Fraude'], rotation=0)

# Gráfico de pizza
df['Class'].value_counts().plot(kind='pie', ax=axes[1], labels=['Legítimo', 'Fraude'], 
                                  colors=['green', 'red'], autopct='%1.2f%%')
axes[1].set_title('Proporção de Classes')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"Dataset altamente desequilibrado: {df['Class'].value_counts()[0]/len(df)*100:.2f}% legítimos e {df['Class'].value_counts()[1]/len(df)*100:.2f}% fraudes")

In [ ]:
# Análise de correlação
plt.figure(figsize=(12, 10))
correlation = df.corr()
sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False, fmt='.2f')
plt.title('Matriz de Correlação das Variáveis')
plt.tight_layout()
plt.show()

# Correlação com a classe
print("\nTop 10 variáveis mais correlacionadas com fraude:")
print(correlation['Class'].sort_values(ascending=False)[1:11])

## 4. Pré-processamento de Dados

In [ ]:
# Separar features e target
X = df.drop('Class', axis=1)
y = df['Class']

# Dividir em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Tamanho do conjunto de treino: {X_train.shape}")
print(f"Tamanho do conjunto de teste: {X_test.shape}")
print(f"\nDistribuição no treino:")
print(y_train.value_counts())
print(f"\nDistribuição no teste:")
print(y_test.value_counts())

In [ ]:
# Normalizar as features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dados normalizados com sucesso!")
print(f"Média das features de treino: {X_train_scaled.mean(axis=0)[:5]}") # Primeiras 5
print(f"Desvio padrão das features de treino: {X_train_scaled.std(axis=0)[:5]}") # Primeiras 5

## 5. Construção e Treinamento dos Modelos

In [ ]:
# Dicionário para armazenar modelos
modelos = {}
resultados = {}

# Modelo 1: Regressão Logística
print("Treinando Regressão Logística...")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
modelos['Regressão Logística'] = lr
print("✓ Concluído")

# Modelo 2: Random Forest
print("\nTreinando Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
modelos['Random Forest'] = rf
print("✓ Concluído")

# Modelo 3: Gradient Boosting
print("\nTreinando Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
modelos['Gradient Boosting'] = gb
print("✓ Concluído")

## 6. Avaliação dos Modelos

In [ ]:
# Fazer previsões
print("="*80)
print("AVALIAÇÃO DOS MODELOS")
print("="*80)

for nome, modelo in modelos.items():
    print(f"\n{'='*80}")
    print(f"Modelo: {nome}")
    print(f"{'='*80}")
    
    # Previsões
    if nome == 'Regressão Logística':
        y_pred = modelo.predict(X_test_scaled)
        y_pred_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = modelo.predict(X_test)
        y_pred_proba = modelo.predict_proba(X_test)[:, 1]
    
    # Métricas
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    
    resultados[nome] = {'ROC-AUC': roc_auc, 'F1': f1}
    
    print(f"\nROC-AUC Score: {roc_auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    print(f"\nMatriz de Confusão:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    print(f"\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred, target_names=['Legítimo', 'Fraude']))

In [ ]:
# Comparação de modelos
df_resultados = pd.DataFrame(resultados).T
print("\n" + "="*80)
print("RESUMO COMPARATIVO DOS MODELOS")
print("="*80)
print(df_resultados)

# Visualizar comparação
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_resultados['ROC-AUC'].plot(kind='bar', ax=axes[0], color=['blue', 'green', 'red'])
axes[0].set_title('Comparação ROC-AUC')
axes[0].set_ylabel('ROC-AUC Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].set_ylim([0, 1])

df_resultados['F1'].plot(kind='bar', ax=axes[1], color=['blue', 'green', 'red'])
axes[1].set_title('Comparação F1 Score')
axes[1].set_ylabel('F1 Score')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 7. Curvas ROC e Precision-Recall

In [ ]:
# Curvas ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nome, modelo in modelos.items():
    if nome == 'Regressão Logística':
        y_pred_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred_proba = modelo.predict_proba(X_test)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    axes[0].plot(fpr, tpr, label=f'{nome} (AUC = {roc_auc:.3f})', linewidth=2)
    
    # Precision-Recall
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    f1 = f1_score(y_test, (y_pred_proba > 0.5).astype(int))
    axes[1].plot(recall, precision, label=f'{nome} (F1 = {f1:.3f})', linewidth=2)

# Configurar gráfico ROC
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Classificador Aleatório')
axes[0].set_xlabel('Taxa de Falsos Positivos')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos')
axes[0].set_title('Curva ROC')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Configurar gráfico Precision-Recall
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Análise de Importância das Features

In [ ]:
# Importância das features - Random Forest
importances_rf = modelos['Random Forest'].feature_importances_
feature_names = X.columns
indices_rf = np.argsort(importances_rf)[-10:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
axes[0].barh(feature_names[indices_rf], importances_rf[indices_rf])
axes[0].set_xlabel('Importância')
axes[0].set_title('Top 10 Features - Random Forest')

# Gradient Boosting
importances_gb = modelos['Gradient Boosting'].feature_importances_
indices_gb = np.argsort(importances_gb)[-10:]
axes[1].barh(feature_names[indices_gb], importances_gb[indices_gb])
axes[1].set_xlabel('Importância')
axes[1].set_title('Top 10 Features - Gradient Boosting')

plt.tight_layout()
plt.show()

## 9. Conclusões e Recomendações

In [ ]:
print("\n" + "="*80)
print("CONCLUSÕES E RECOMENDAÇÕES")
print("="*80)

melhor_modelo = df_resultados['ROC-AUC'].idxmax()
melhor_auc = df_resultados['ROC-AUC'].max()

print(f"\n✓ Melhor modelo: {melhor_modelo}")
print(f"  - ROC-AUC Score: {melhor_auc:.4f}")
print(f"  - F1 Score: {df_resultados.loc[melhor_modelo, 'F1']:.4f}")

print("\nRecomendações:")
print("1. O modelo selecionado apresenta excelente performance na detecção de fraudes.")
print("2. Considerar ajuste de threshold para otimizar precision/recall conforme necessário.")
print("3. Implementar monitoramento contínuo do modelo em produção.")
print("4. Realizar reciclagem periódica do modelo com novos dados.")
print("5. Considerar técnicas de balanceamento de classes para melhorar performance.")